# Indian Air Quality Forecaster Model Training
**Dataset**: CPCB National Air Quality Dataset (city_day.csv)
**Goal**: Predict next-day AQI and severity level with high accuracy

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_absolute_error, classification_report
print("Libraries imported successfully!")


In [ ]:
import glob
import os
print("Available input directories:", os.listdir("/kaggle/input"))
matching = glob.glob("/kaggle/input/**/city_day.csv", recursive=True) or glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("Found files:", matching)
if not matching:
    raise FileNotFoundError("No city_day.csv found in /kaggle/input")
data_path = matching[0]
print(f"Reading CPCB dataset from {data_path}")
df = pd.read_csv(data_path)
print(f"Raw dataset shape: {df.shape}")
print(df.head())


In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["City", "Date"]).reset_index(drop=True)

# Drop rows without AQI
df = df.dropna(subset=["AQI"]).copy()

# Feature engineering: date features
df["month"] = df["Date"].dt.month
df["day_of_year"] = df["Date"].dt.dayofyear
df["day_of_week"] = df["Date"].dt.dayofweek

# Group by City to create true temporal lags
for col in ["AQI", "PM2.5", "PM10", "NO2", "SO2", "CO"]:
    df[f"{col}_lag1"] = df.groupby("City")[col].shift(1)
    df[f"{col}_lag2"] = df.groupby("City")[col].shift(2)
    df[f"{col}_roll3"] = df.groupby("City")[col].transform(lambda x: x.rolling(3, min_periods=1).mean())

# Target: Next-day AQI Bucket classification
# Standard Indian NAQI 6 categories: 0=Good, 1=Satisfactory, 2=Moderate, 3=Poor, 4=Very Poor, 5=Severe
def categorize_aqi(val):
    if val <= 50: return 0
    elif val <= 100: return 1
    elif val <= 200: return 2
    elif val <= 300: return 3
    elif val <= 400: return 4
    else: return 5

df["target_bucket"] = df.groupby("City")["AQI"].shift(-1).apply(lambda v: categorize_aqi(v) if pd.notnull(v) else np.nan)
df = df.dropna(subset=["target_bucket", "AQI_lag1"]).copy()
df["target_bucket"] = df["target_bucket"].astype(int)

print(f"Processed dataset shape: {df.shape}")
print("Category distribution:")
print(df["target_bucket"].value_counts().sort_index())


In [ ]:
features = [
    "AQI_lag1", "AQI_lag2", "AQI_roll3",
    "PM2.5_lag1", "PM2.5_roll3", "PM10_lag1", "PM10_roll3",
    "NO2_lag1", "SO2_lag1", "CO_lag1",
    "month", "day_of_year", "day_of_week"
]

# Fill remaining feature NaNs with column medians
for col in features:
    df[col] = df[col].fillna(df[col].median())

# Temporal train/test split: Train before 2019, Test 2019-2020
train_mask = df["Date"] < "2019-01-01"
test_mask = df["Date"] >= "2019-01-01"

X_train, y_train = df.loc[train_mask, features], df.loc[train_mask, "target_bucket"]
X_test, y_test = df.loc[test_mask, features], df.loc[test_mask, "target_bucket"]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


In [ ]:
print("Training XGBoost Multi-Class Classifier for AQI category forecast...")
model = xgb.XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    num_class=6,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
within_1_acc = np.mean(np.abs(y_test - y_pred) <= 1)
f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 50)
print(f"EXACT ACCURACY:              {acc * 100:.2f}%")
print(f"WITHIN-1 CATEGORY ACCURACY:  {within_1_acc * 100:.2f}%")
print(f"WEIGHTED F1-SCORE:           {f1:.4f}")
print("=" * 50)
print(classification_report(y_test, y_pred))


In [ ]:
model.save_model("/kaggle/working/aqi_xgb_v1.json")
metrics = {
    "model_name": "aqi_xgb_v1",
    "exact_accuracy": round(acc, 4),
    "within_1_accuracy": round(within_1_acc, 4),
    "weighted_f1": round(f1, 4),
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "features": features
}
with open("/kaggle/working/aqi_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved aqi_xgb_v1.json and aqi_metrics.json to /kaggle/working/")
